In [1]:
import pandas as pd
import tensorflow as tf

In [9]:
df = pd.read_csv('https://drive.google.com/uc?id=1AZRfFoyekqSYpri5183RmJjciRGz_ood', sep=',', index_col='datetime', header=0)
df

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...
2007-02-14 17:19:00,0.636,0.140,241.16,2.6,0.0,0.0,0.0
2007-02-14 17:20:00,0.552,0.000,240.46,2.2,0.0,0.0,0.0
2007-02-14 17:21:00,0.538,0.000,239.74,2.2,0.0,0.0,0.0


### preprocessing

In [10]:
def normalize_series(data, min, max):
  data = data - min
  data = data / max
  return data
data = df.values
data = normalize_series(data, data.min(axis=0), data.max(axis=0))

panjang data

In [12]:
N_FEATURES = len(df.columns)

splitting 50:50

In [13]:
SPLIT_TIME = int(len(data) * 0.5)
x_train = data[:SPLIT_TIME]
x_valid = data[SPLIT_TIME:]

windowing

In [14]:
def windowed_dataset(series, batch_size, n_past=24, n_future=24, shift=1):
  ds = tf.data.Dataset.from_tensor_slices(series)
  ds = ds.window(size=n_past + n_future, shift=shift, drop_remainder=True)
  ds = ds.flat_map(lambda w: w.batch(n_past + n_future))
  ds = ds.map(lambda w: (w[:n_past], w[n_past:]))
  return ds.batch(batch_size).prefetch(1)

In [15]:
BATCH_SIZE = 32
N_PAST = 24
N_FUTURE = 24
SHIFT = 1
# Kode untuk membuat windowed datasets
train_set = windowed_dataset(series=x_train, batch_size=BATCH_SIZE, 
                             n_past=N_PAST, n_future=N_FUTURE,
                             shift=SHIFT)
valid_set = windowed_dataset(series=x_valid, batch_size=BATCH_SIZE,
                             n_past=N_PAST, n_future=N_FUTURE,
                             shift=SHIFT)

### Arsitektur model
2 layer dense

In [16]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Dense(64, input_shape=(N_PAST, N_FEATURES)),
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dense(N_FEATURES)
])

c:\Users\Thinkpad\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### hyperparameter mycallback
pelatihan akan berhenti ketika tingkat kinerja model telah mencapai tingkat yang dianggap memadai, yaitu ketika nilai MAE pada data pelatihan dan data validasi sudah cukup rendah.

In [17]:
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
      if (logs.get('mae') < 0.055 and logs.get('val_mae') < 0.055):
         self.model.stop_training = True

callbacks = myCallback()

### metrik evaluasi

In [19]:
# Kode untuk melakukan menyusun struktur sesuai dengan machine learning
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss='mae',
              optimizer= optimizer,
              metrics=["mae"])

## Pelatihan

In [20]:
model.fit(train_set,
          validation_data=(valid_set),
          epochs=100,
          callbacks=callbacks,
          verbose=1
          )

Epoch 1/100
   1348/Unknown 17s 11ms/step - loss: 0.0822 - mae: 0.0822

c:\Users\Thinkpad\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1349/1349 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/step - loss: 0.0682 - mae: 0.0682 - val_loss: 0.0615 - val_mae: 0.0615
Epoch 2/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 25s 19ms/step - loss: 0.0602 - mae: 0.0602 - val_loss: 0.0571 - val_mae: 0.0571
Epoch 3/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - loss: 0.0590 - mae: 0.0590 - val_loss: 0.0582 - val_mae: 0.0582
Epoch 4/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 21s 16ms/step - loss: 0.0587 - mae: 0.0587 - val_loss: 0.0581 - val_mae: 0.0581
Epoch 5/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 31s 23ms/step - loss: 0.0586 - mae: 0.0586 - val_loss: 0.0605 - val_mae: 0.0605
Epoch 6/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - loss: 0.0584 - mae: 0.0584 - val_loss: 0.0596 - val_mae: 0.0596
Epoch 7/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 29s 21ms/step - loss: 0.0580 - mae: 0.0580 - val_loss: 0.0579 - val_mae: 0.0579
Epoch 8/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 29s 21ms/step - loss: 0.0578 - mae: 0.0578 - val_loss: 0.0598 - val_mae: 0.0598
Epoch 9/100
1349/1349 ━━━━━━

## Prediksi

In [22]:
train_pred = model.predict(train_set)
train_pred[0][0]

1349/1349 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step


array([0.35814303, 0.24883774, 0.02978188, 0.35234845, 0.00191246,
       0.00388249, 0.8295334 ], dtype=float32)